In [6]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import time
from pathlib import Path
import sys

# ==========================================
# 0. PARAMÈTRES DE TON ÉQUIPE ET ENVIRONNEMENT
# ==========================================
if 'google.colab' in sys.modules:
    print("☁️ Environnement Colab détecté. Connexion au Drive...")
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = "/content/drive/MyDrive/Documents/Code/MonPetitPronoStrategy/"
else:
    print("💻 Environnement Local (VS Code) détecté.")
    PROJECT_DIR = str(Path.cwd().parent) + "/"
sys.path.append(PROJECT_DIR)

from mpp_project.v5_supervised.bracket_simulator import generate_bracket_scenario
from mpp_project.v5_supervised.oracle_dp import rescale_histogram_1D, compute_full_Q_table
from mpp_project.v5_supervised.end_game_solver import solve_endgame_dp, GAP_MIN, GAP_MAX, GAP_OFFSET, GRID_SIZE

# --- PARAMÈTRES DU JOUR ---
# Renseigne l'ID officiel du match (ex: 73 pour le premier 16ème de finale)
MATCH_DU_JOUR_ID = 73    
match_idx_global = MATCH_DU_JOUR_ID - 1

# 🚀 NOUVEAU : Conversion pour l'arbre des phases finales (0 à 31)
# On part du principe qu'il y a 72 matchs de poule (0 à 71)
match_idx_pf = match_idx_global - 72 

MON_GAP_1 = 0
MON_GAP_2 = 0
HAS_BOOSTER = 1  # 1 = Booster dispo, 0 = Déjà utilisé
HORIZON_NUIT = 4 # Nombre de matchs à exporter pour l'appli mobile

# --- ÉTAT DES FAVORIS ET BUTEURS (V_term) ---
my_fav_alive = 1 
bob_fav_alive = 1
pack_fav_alive = 1

print(f"🎯 Match cible Global : {MATCH_DU_JOUR_ID} | Index Python Phase Finale : {match_idx_pf}")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
💻 Environnement Local (VS Code) détecté.
🎯 Match cible Global : 73 | Index Python Phase Finale : 0


In [7]:
# ==========================================
# 1. PRÉPARATION DU BRACKET ET RESCALING
# ==========================================
print("Chargement des cotes du jour depuis le fichier global...")
df_full = pd.read_csv(f"{PROJECT_DIR}data/CDM_2026.csv")

# 🚀 NOUVEAU : On isole les 32 matchs des phases finales en excluant les poules
df_pf = df_full[~df_full['phase'].str.contains('Poule', na=False)].reset_index(drop=True)

if len(df_pf) != 32:
    print(f"⚠️ ATTENTION : Le DataFrame filtré contient {len(df_pf)} matchs au lieu de 32 !")

# On passe uniquement les phases finales au simulateur
match_probs, gains_arbre_actuel, crowds, match_teams, team_to_id = generate_bracket_scenario(df_pf)

print("Chargement de la Thermodynamique de Référence...")
p_emp_ref = np.load(f"{PROJECT_DIR}data/peloton_reference_histograms.npy")
gains_ref = np.load(f"{PROJECT_DIR}data/peloton_reference_gains.npy")

# Rescaling Magique (On utilise match_idx_pf !)
p_empirique_scenario = np.zeros((32, 3, 250), dtype=np.float32)
for t in range(match_idx_pf, 32):
    p1, p2 = match_probs[t, 0], match_probs[t, 2]
    idx_fav = 0 if p1 >= p2 else 2
    mapping = {idx_fav: 0, 1: 1, (2 if idx_fav == 0 else 0): 2}
    
    for out in range(3):
        p_empirique_scenario[t, out] = rescale_histogram_1D(
            ref_hist=p_emp_ref[t, mapping[out]],
            ref_gain=gains_ref[t, mapping[out]],
            target_gain=gains_arbre_actuel[t, out],
            target_crowd=crowds[t, out],
            max_bins=250
        )

# ==========================================
# 2. RÉSOLUTION DE LA DP JUSQU'AU MATCH DU JOUR
# ==========================================
print("\nLancement de l'Oracle DP (Phases Finales)...")
start_dp = time.time()

alphas_mock = np.ones(32, dtype=np.float32)
V_term = np.load(f"{PROJECT_DIR}data/V_term_buteurs.npy") 

# L'Oracle s'arrête à match_idx_pf
V_all_matches = solve_endgame_dp(
    match_probs, crowds, gains_arbre_actuel, p_empirique_scenario, alphas_mock,
    V_term=V_term, 
    stop_t=match_idx_pf
)
print(f"-> Résolution DP terminée en {time.time() - start_dp:.2f} s.")

# ==========================================
# 3. EXTRACTION ET EXPORT DE L'ABAQUE (Q-TABLE)
# ==========================================
print("\n" + "="*50)
print(f"Génération de l'Abaque pour téléphone...")
print("="*50)

for k in range(HORIZON_NUIT):
    t = match_idx_pf + k
    if t >= 32: break
        
    # On recalcule l'ID global pour l'affichage (ex: Abaque_Match_73)
    match_id_cible_global = t + 73 
    
    V_next_t = V_all_matches[t + 1] if (t + 1) < 32 else V_term

    t_prob = match_probs[t]
    c_rep = crowds[t]
    t_gains = gains_arbre_actuel[t]
    out_p_empirique = p_empirique_scenario[t]

    print(f"Calcul de l'Abaque pour le Match {match_id_cible_global}...")
    
    Q_table_t = compute_full_Q_table(
        t=t, t_prob=t_prob, c_rep=c_rep, t_gains=t_gains, 
        p_empirique_1D=out_p_empirique, alpha=1.0,
        V_next=V_next_t, max_gain=250 
    )

    export_path = f"{PROJECT_DIR}data/Abaque_Match_{match_id_cible_global}.npz"
    np.savez_compressed(export_path, q_table=Q_table_t, ev_actions=t_prob * t_gains)
    
print(f"\n✅ Terminé ! {HORIZON_NUIT} fichiers synchronisés sur le Drive.")
print("Le notebook '17_night_mode.ipynb' est désormais utilisable.")

Chargement des cotes du jour depuis le fichier global...
⚠️ ATTENTION : Le DataFrame filtré contient 16 matchs au lieu de 32 !


KeyError: 'group'

In [ ]:
# ==========================================
# 4. PROJECTION STRATÉGIQUE SUR LES PROCHAINS MATCHS
# ==========================================
print("\n" + "="*50)
print("🔮 PROJECTION STRATÉGIQUE (PHASES FINALES)")
print("="*50)
print("Analyse de sensibilité : Évolution de la stratégie selon la dynamique\n")

noms_choix = ["1 (Dom)", "N (Nul)", "2 (Ext)"]
scenarios = [
    {"nom": "🔴 Retard (-30 pts/match)", "delta": -30},
    {"nom": "⚪ Maintien (Inchangé)", "delta": 0},
    {"nom": "🟢 Avance (+30 pts/match)", "delta": 30}
]

horizon_projection = min(4, HORIZON_NUIT, 32 - match_idx)

for k in range(horizon_projection):
    t = match_idx + k
    match_id_cible = t + 1
    
    try:
        abaque_path = f"{PROJECT_DIR}data/Abaque_Match_PF_{match_id_cible}.npz"
        data = np.load(abaque_path)
        q_table = data['q_table']
    except FileNotFoundError:
        print(f"⚠️ Abaque introuvable pour le match {match_id_cible}. Fin de la projection.")
        break
        
    print(f"▶️ MATCH {match_id_cible} (Δt = +{k}) :")
    
    for sc in scenarios:
        proj_gap1 = MON_GAP_1 + (sc["delta"] * k)
        proj_gap2 = MON_GAP_2 + (sc["delta"] * k)
        
        g1_idx = max(GAP_MIN, min(GAP_MAX, int(round(proj_gap1)))) + GAP_OFFSET
        g2_idx = max(GAP_MIN, min(GAP_MAX, int(round(proj_gap2)))) + GAP_OFFSET
        
        if HAS_BOOSTER:
            wr_keep = q_table[g1_idx, g2_idx, :, 1]
            wr_use = q_table[g1_idx, g2_idx, :, 2]
            
            best_keep_idx = np.argmax(wr_keep)
            best_use_idx = np.argmax(wr_use)
            
            val_keep = wr_keep[best_keep_idx]
            val_use = wr_use[best_use_idx]
            
            if val_use > val_keep:
                reco = f"{noms_choix[best_use_idx]} + x2"
            else:
                reco = f"{noms_choix[best_keep_idx]} (Safe)"
                
            details_wr = f"WR(Safe): {val_keep*100:05.2f}% | WR(x2): {val_use*100:05.2f}%"
        else:
            wr_base = q_table[g1_idx, g2_idx, :, 0]
            best_action = np.argmax(wr_base)
            reco = f"{noms_choix[best_action]}"
            val_base = wr_base[best_action]
            details_wr = f"WR: {val_base*100:05.2f}%"
            
        print(f"  {sc['nom']:<27} | Gaps proj. : {proj_gap1:>4}, {proj_gap2:>4} | Reco : {reco:<14} | {details_wr}")
    print("-" * 90)